# Logistic regression with different feature sets

In [28]:
%load_ext autoreload
%autoreload 2
import constants
import pandas as pd
import pm4py
import ast
from helper_enriching import add_case_attribute, get_month_attribute, get_weekday_attribute, get_urheber_first_activity, get_urheber_count_first_activity, get_WIP_during_start, get_is_passed_bill, get_is_election_year, get_half_year_attribute, get_government_party_count_attribute, get_opposition_party_count_attribute, get_opposition_seat_percentage_attribute, get_government_seat_percentage_attribute, add_pdf_case_attributes, get_pdf_information
from sklearn.preprocessing import MultiLabelBinarizer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Enrich event logs with context features

## Read Data

In [29]:
'''
INPUT_FILE_NAME = "./data-xes/Berlin-with-topics-and-agreement-scores-all-time.xes"
OUTPUT_FILE_NAME = "./data-csv/case-logs/Berlin-with-topics-and-agreement-scores-all-time.csv"
BUNDESLAND = "berlin"
electionPeriodStartDates = constants.BERLIN_ELECTION_PERIODS_START_DATES
coalitionsSorted = constants.BERLIN_COALITIONS_SORTED

'''
INPUT_FILE_NAME = "./data-xes/Brandenburg-with-topics-and-agreement-scores-all-time-preprocessed.xes"
OUTPUT_FILE_NAME = "./data-csv/case-logs/Brandenburg-with-topics-and-agreement-scores-all-time-preprocessed.csv"
BUNDESLAND = "brandenburg"
electionPeriodStartDates = constants.BRANDENBURG_ELECTION_PERIODS_START_DATES
coalitionsSorted = constants.BRANDENBURG_COALITIONS_SORTED


'''
INPUT_FILE_NAME = "./data-xes/Baden-Württemberg-with-topics-and-agreement-scores-all-time-preprocessed.xes"
OUTPUT_FILE_NAME = "./data-csv/case-logs/Baden-Württemberg-with-topics-and-agreement-scores-all-time-preprocessed.csv"
BUNDESLAND = "baden-württemberg"
electionPeriodStartDates = constants.BAWUE_ELECTION_PERIODS_START_DATES
coalitionsSorted = constants.BADEN_WUERTTEMBERG_COALITIONS_SORTED
'''



CASE_ID_COL = 'case:concept:name'
TIMESTAMP_COL = 'time:timestamp'
ACTIVITY_COL = 'concept:name'

df = pm4py.read_xes(INPUT_FILE_NAME)
df = df.sort_values(TIMESTAMP_COL, ignore_index=True)
print(len(df), 'rows read from', INPUT_FILE_NAME)

parsing log, completed traces ::   0%|          | 0/1410 [00:00<?, ?it/s]

13890 rows read from ./data-xes/Brandenburg-with-topics-and-agreement-scores-all-time-preprocessed.xes


In [30]:
# remove columns that are not of interest at all
columns_to_drop = ['DBID', 'ReihNr', 'DokArt', 'DokArtL', 'lifecycle:transition', 
                   'Titel', 'DokNr', 'Sb', 'HNr', 'Jg', 'NrInTyp', 
                   'DHerk', 'DHerkL', 'DokTyp', '@@index', 'DokDat', 'VkDat',
                   'case:variant', 'case:variant-index', 'case:creator',
                    'case:VSys',
                    'case:VTyp', 'case:VTypL',
                   'case:DHerk', 'case:DHerkL', 'case:NrInTyp',
                   'case:@@case_index',
                   "case:embedding",
                   "case:top_topics",
                   #'LokURL',
                   ]
df = df.drop(columns=columns_to_drop, errors='ignore')

## Enrich with features from before

In [31]:
#add_case_attribute(df, get_month_attribute(df), 'case:start_month')
#add_case_attribute(df, get_weekday_attribute(df), "case:start_weekday")
add_case_attribute(df, get_WIP_during_start(df), "case:WIP_during_start")
add_case_attribute(df, get_is_passed_bill(df), "case:is_passed_bill")
add_case_attribute(df, get_is_election_year(df, bundesland=BUNDESLAND), "case:is_election_year")


In [32]:
pdf_info_bytes, pdf_info_word_count = get_pdf_information(df)
add_pdf_case_attributes(df, pdf_info_bytes, pdf_info_word_count, "case:pdf_bytes", "case:pdf_word_count")

i/len(df): 0 / 13890
getting file...
got file...
{'bytes': 19330, 'total_word_count': 1079}
i/len(df): 1 / 13890
getting file...
got file...
{'bytes': 28303, 'total_word_count': 311}
i/len(df): 2 / 13890
i/len(df): 3 / 13890
i/len(df): 4 / 13890
getting file...
got file...
{'bytes': 31293, 'total_word_count': 528}
i/len(df): 5 / 13890
getting file...
got file...
{'bytes': 125515, 'total_word_count': 8714}
i/len(df): 6 / 13890
i/len(df): 7 / 13890
i/len(df): 8 / 13890
i/len(df): 9 / 13890
i/len(df): 10 / 13890
i/len(df): 11 / 13890
i/len(df): 12 / 13890
i/len(df): 13 / 13890
getting file...
got file...
{'bytes': 63067, 'total_word_count': 3308}
i/len(df): 14 / 13890
getting file...
got file...
{'bytes': 112557, 'total_word_count': 14418}
i/len(df): 15 / 13890
i/len(df): 16 / 13890
i/len(df): 17 / 13890
i/len(df): 18 / 13890
getting file...
got file...
{'bytes': 80605, 'total_word_count': 4421}
i/len(df): 19 / 13890
getting file...
got file...
{'bytes': 97839, 'total_word_count': 4204}
i

## Further political context features

#### Stuff already in the data and/or added before already: 
- Urheber, Systematiken (case:VSysL), Deskriptoren (case:VorgangsDeskriptoren)
- Wahl-O-Mat agreement scores

In [33]:
## Urheber 

#add_case_attribute(df, get_urheber_first_activity(df), "case:author_first_event") # ToDo: needs to be encoded
add_case_attribute(df, get_urheber_count_first_activity(df), "case:author_first_event_count")


def parse_urheber(val):
    """Parse the Urheber field into a list of normalized names."""
    if not isinstance(val, str):
        return []
    if val.startswith("[") and val.endswith("]"):
        items = ast.literal_eval(val)
    else:
        items = [val]
    return ["ausschuss" if "ausschuss" in u.lower() else u.lower() for u in items]


df["urheber_parsed"] = df["Urheber"].apply(parse_urheber)

# check how many unique urheber exist
case_urheber = df.groupby(CASE_ID_COL)["urheber_parsed"].first().explode()
label_counts_urheber = case_urheber.value_counts()

print(f"Unique labels: {len(label_counts_urheber)}")
print(f"Mean: {label_counts_urheber.mean():.1f}, Median: {label_counts_urheber.median():.1f}")

for threshold in [5, 10, 20, 30, 50]:
    n_kept = (label_counts_urheber >= threshold).sum()
    print(f"Threshold {threshold}: {n_kept} features kept")

MIN_COUNT_URHEBER = 10

# Identify frequent labels
frequent_urheber = set(label_counts_urheber[label_counts_urheber >= MIN_COUNT_URHEBER].index)

def parse_urheber_grouped(val):
    labels = parse_urheber(val)
    return [l if l in frequent_urheber else "sonstige" for l in labels]

# Re-parse with grouping
df["urheber_parsed"] = df["Urheber"].apply(parse_urheber_grouped)

# Get the first activity per case
first_activities = df.groupby(CASE_ID_COL)["urheber_parsed"].first()

# One-hot encode
mlb = MultiLabelBinarizer()
encoded = pd.DataFrame(
    mlb.fit_transform(first_activities),
    index=first_activities.index,
    columns=[f"case:author_first_event_{c}" for c in mlb.classes_]
)

# Merge the one-hot encoded features back into the main DataFrame
encoded_per_row = df[[CASE_ID_COL]].merge(encoded, left_on=CASE_ID_COL, right_index=True, how="left")
for col in encoded.columns:
    df[col] = encoded_per_row[col].values

df = df.drop(columns=["urheber_parsed"])



Unique labels: 107
Mean: 16.3, Median: 1.0
Threshold 5: 27 features kept
Threshold 10: 15 features kept
Threshold 20: 11 features kept
Threshold 30: 10 features kept
Threshold 50: 5 features kept


In [34]:
# Systematiken

def parse_vsysl(val):
    if not isinstance(val, str):
        return []
    if val.startswith("[") and val.endswith("]"):
        return [v.strip() for v in ast.literal_eval(val)]
    return [val]

# Count per case (not per event)
case_vsysl = df.groupby(CASE_ID_COL)["case:VSysL"].first().apply(parse_vsysl).explode()
label_counts = case_vsysl.value_counts()

MIN_COUNT = 30 # ToDo: adjust maybe based on dataset size??
print(f"Features kept: {(label_counts >= MIN_COUNT).sum()} of {len(label_counts)}")

# Or try a few thresholds to see the tradeoff
for threshold in [5, 10, 20, 30, 50]:
    n_kept = (label_counts >= threshold).sum()
    print(f"Threshold {threshold}: {n_kept} features kept")

frequent_labels = set(label_counts[label_counts >= MIN_COUNT].index)


def parse_vsysl_grouped(val):
    labels = parse_vsysl(val)
    return [l if l in frequent_labels else "sonstige" for l in labels]

df["vsysl_parsed"] = df["case:VSysL"].apply(parse_vsysl_grouped)

first_vsysl = df.groupby(CASE_ID_COL)["vsysl_parsed"].first()

mlb_vsysl = MultiLabelBinarizer()
encoded_vsysl = pd.DataFrame(
    mlb_vsysl.fit_transform(first_vsysl),
    index=first_vsysl.index,
    columns=[f"case:vsysl_{c}" for c in mlb_vsysl.classes_]
)

# Merge back into df
encoded_vsysl_per_row = df[[CASE_ID_COL]].merge(encoded_vsysl, left_on=CASE_ID_COL, right_index=True, how="left")
for col in encoded_vsysl.columns:
    df[col] = encoded_vsysl_per_row[col].values

df = df.drop(columns=["vsysl_parsed"])


Features kept: 11 of 132
Threshold 5: 82 features kept
Threshold 10: 49 features kept
Threshold 20: 17 features kept
Threshold 30: 11 features kept
Threshold 50: 4 features kept


In [35]:
# Vorgangsdeskriptoren

def parse_deskriptoren(val):
    if not isinstance(val, str):
        return []
    if val.startswith("[") and val.endswith("]"):
        return [v.strip() for v in ast.literal_eval(val)]
    return [val]

case_deskriptoren = df.groupby(CASE_ID_COL)["case:VorgangsDeskriptoren"].first().apply(parse_deskriptoren).explode()
label_counts_desk = case_deskriptoren.value_counts()

print(f"Unique labels: {len(label_counts_desk)}")
print(f"Mean: {label_counts_desk.mean():.1f}, Median: {label_counts_desk.median():.1f}")

for threshold in [5, 10, 20, 30, 50]:
    n_kept = (label_counts_desk >= threshold).sum()
    print(f"Threshold {threshold}: {n_kept} features kept")

MIN_COUNT_DESK = 30  # adjust after seeing the output above

frequent_desk = set(label_counts_desk[label_counts_desk >= MIN_COUNT_DESK].index)

def parse_deskriptoren_grouped(val):
    labels = parse_deskriptoren(val)
    return [l if l in frequent_desk else "sonstige" for l in labels]

df["deskriptoren_parsed"] = df["case:VorgangsDeskriptoren"].apply(parse_deskriptoren_grouped)

first_desk = df.groupby(CASE_ID_COL)["deskriptoren_parsed"].first()

mlb_desk = MultiLabelBinarizer()
encoded_desk = pd.DataFrame(
    mlb_desk.fit_transform(first_desk),
    index=first_desk.index,
    columns=[f"case:deskriptor_{c}" for c in mlb_desk.classes_]
)

# Merge back
encoded_desk_per_row = df[[CASE_ID_COL]].merge(encoded_desk, left_on=CASE_ID_COL, right_index=True, how="left")
for col in encoded_desk.columns:
    df[col] = encoded_desk_per_row[col].values

df = df.drop(columns=["deskriptoren_parsed"])

Unique labels: 3963
Mean: 3.3, Median: 2.0
Threshold 5: 689 features kept
Threshold 10: 247 features kept
Threshold 20: 67 features kept
Threshold 30: 25 features kept
Threshold 50: 8 features kept


In [36]:
'''
print(df.columns)

'DokTypL', 'Desk', 'Urheber', 'Abstract', 'Redner', 'DokDat_original',
       'top_topics', 'top_topic', 'top_topic_similarity_score',
       'government_agreement_score', 'opposition_agreement_score',
       'all_agreement_score', 'concept:name', 'time:timestamp', '@@case_index',
       'case:concept:name', 'case:DokTypLFirstDoc', 'case:VSysL',
       'case:VorgangsDeskriptoren', 'case:Wp', 'case:embedding',
       'case:start_month', 'case:start_weekday', 'case:author_first_activity',
       'case:author_first_activity_count', 'case:WIP_during_start',
       'case:is_passed_bill', 'case:is_election_year'
'''

"\nprint(df.columns)\n\n'DokTypL', 'Desk', 'Urheber', 'Abstract', 'Redner', 'DokDat_original',\n       'top_topics', 'top_topic', 'top_topic_similarity_score',\n       'government_agreement_score', 'opposition_agreement_score',\n       'all_agreement_score', 'concept:name', 'time:timestamp', '@@case_index',\n       'case:concept:name', 'case:DokTypLFirstDoc', 'case:VSysL',\n       'case:VorgangsDeskriptoren', 'case:Wp', 'case:embedding',\n       'case:start_month', 'case:start_weekday', 'case:author_first_activity',\n       'case:author_first_activity_count', 'case:WIP_during_start',\n       'case:is_passed_bill', 'case:is_election_year'\n"

In [37]:
# Half-year number within election period
half_year_obj = get_half_year_attribute(df, electionPeriodStartDates)
add_case_attribute(df, half_year_obj, "case:half_year_index")

gov_count = get_government_party_count_attribute(df, electionPeriodStartDates, coalitionsSorted)
add_case_attribute(df, gov_count, "case:government_party_count")

opp_count = get_opposition_party_count_attribute(df, electionPeriodStartDates, coalitionsSorted)
add_case_attribute(df, opp_count, "case:opposition_party_count")


# ToDo: Probably this would make more sense to have only the percentage of seats for this specific traces Urheber...
opp_seat_perc = get_opposition_seat_percentage_attribute(df, electionPeriodStartDates, coalitionsSorted)
add_case_attribute(df, opp_seat_perc, "case:opposition_seat_percentage")

gov_seat_perc = get_government_seat_percentage_attribute(df, electionPeriodStartDates, coalitionsSorted)
add_case_attribute(df, gov_seat_perc, "case:government_seat_percentage")

## Case Log Creation and Enrichment from Völzer et al.:

In [38]:
# Configure the number of attributes to be included
INCLUDE_DATA = False
INCLUDE_TIME = True
INCLUDE_DELAYS = True
INCLUDE_DATA_NOT_CONNECTED_TO_ACTIVITIES = True


case_index = {}
activity_list = []
log_starting_time = df.at[0, TIMESTAMP_COL] # this will be used to calculate the relative time
columns = df.columns.to_list()

for col in [CASE_ID_COL, ACTIVITY_COL, TIMESTAMP_COL]:
    columns.remove(col)
    
for index, row in df.iterrows():
    case_id = row[CASE_ID_COL]
    activity_name = row[ACTIVITY_COL]
    
    if not activity_name in activity_list:
        activity_list.append(activity_name)
        
    qualifier = activity_name + "."
    
    if not case_id in case_index:
        case_record = {'case_id': case_id, qualifier+'count': 1, 'event_count': 1}
        
        # add time related features
        if INCLUDE_TIME:
            case_record['start_time'] = row[TIMESTAMP_COL]
            case_record['start_time_rel'] = int((row[TIMESTAMP_COL]-log_starting_time).components.days)
            case_record['duration'] = 0
            case_record[qualifier + 'start'] = 0
        
        
        # add data related features
        for col in columns:
            if not pd.isna(row[col]):
                if INCLUDE_DATA:
                    case_record[qualifier + col] = row[col]
        
        for col in columns:
            if not pd.isna(row[col]):
                if INCLUDE_DATA_NOT_CONNECTED_TO_ACTIVITIES:
                    if (col.startswith('case:')):
                        print(col)
                        case_record[col] = row[col]
                    if (col.endswith("agreement_score")):
                        print(col)
                        case_record[col] = row[col]
        
        # store the information so that it can be updated further 
        # when more events of the same case occur (else part)
        case_index[case_id] = case_record
    
    else:
        case_record = case_index[case_id]
        case_record['event_count'] += 1
        
        # if this activity type has already been seen for this case
        # note that it is the n-th occurrence of this activity type
        # and adjust the qualifier accordingly
        # else set the count to 1
        if qualifier + "start" in case_record:
            activity_count = case_record[qualifier + 'count'] + 1
            case_record[qualifier + 'count'] = activity_count
            qualifier = activity_name + ":" + str(activity_count) + "."
        else:
            case_record[qualifier + 'count'] = 1
        
        if INCLUDE_TIME:
            duration = row[TIMESTAMP_COL] - case_record['start_time']
            case_record[qualifier + 'start'] = int(duration.components.days)
            case_record['duration'] = int(duration.components.days)
        
        
        for col in columns:
            if not pd.isna(row[col]):
                if INCLUDE_DATA:
                    case_record[qualifier + col] = row[col]
        

out_df = pd.DataFrame([case_index[case_id] for case_id in case_index])

# fill nan counts with 0
for col in out_df.columns:
    if col.endswith('.count'):
        out_df[col] = out_df[col].fillna(0)
        
if INCLUDE_DELAYS: 
    first_activities = activity_list.copy()
    while len(first_activities) > 0:
        first_act = first_activities.pop()
        second_activities = first_activities.copy()
        while len(second_activities) > 0:
            second_act = second_activities.pop()
            column_name = first_act + ':' + second_act + '.delay'
            out_df[column_name] = out_df[second_act + '.start'] - out_df[first_act + '.start']

out_df.info(verbose=True)


government_agreement_score
opposition_agreement_score
all_agreement_score
case:DokTypLFirstDoc
case:VSysL
case:VorgangsDeskriptoren
case:WIP_during_start
case:is_passed_bill
case:is_election_year
case:pdf_bytes
case:pdf_word_count
case:author_first_event_count
case:author_first_event_alternative für deutschland (afd)
case:author_first_event_ausschuss
case:author_first_event_brandenburger vereinigte bürgerbewegungen / freie wähler gruppe (bvb / freie wähler gruppe)
case:author_first_event_brandenburger vereinigte bürgerbewegungen / freie wähler gruppe (bvb/fw)
case:author_first_event_bündnis 90 (bü 90)
case:author_first_event_christlich demokratische union deutschlands (cdu)
case:author_first_event_deutsche volksunion (dvu)
case:author_first_event_die linke (die linke)
case:author_first_event_fraktion bündnis 90/die grünen (b90/grüne)
case:author_first_event_freie demokratische partei (f.d.p.)
case:author_first_event_freie demokratische partei (fdp)
case:author_first_event_landesregieru

/var/folders/c0/znjbgdp95852r1d2gl38_cl40000gn/T/ipykernel_14516/2016099581.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out_df[column_name] = out_df[second_act + '.start'] - out_df[first_act + '.start']
/var/folders/c0/znjbgdp95852r1d2gl38_cl40000gn/T/ipykernel_14516/2016099581.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out_df[column_name] = out_df[second_act + '.start'] - out_df[first_act + '.start']
/var/folders/c0/znjbgdp95852r1d2gl38_cl40000gn/T/ipykernel_14516/2016099581.py:98: PerformanceWarning: DataFra

In [39]:
out_df.to_csv(OUTPUT_FILE_NAME, index=False)
print(len(out_df), 'cases written to', OUTPUT_FILE_NAME)

1410 cases written to ./data-csv/case-logs/Brandenburg-with-topics-and-agreement-scores-all-time-preprocessed.csv
